# Failure mining in robotics with FiftyOne

Companion notebook for the blog post *Failure Mining in Robotics: Find One Failure, Surface Every Similar Case*.

You'll find one robot failure in the DROID dataset, record it with a **temporal tag**, narrow the dataset to failed episodes for review, and save the confirmed failures as a reusable set.

- Steps 1, 3 and 4 run in **open source FiftyOne**.
- Step 2, segment-level similarity search, runs on **FiftyOne Enterprise** and is shown as a clearly marked, non-executing section.

In [ ]:
# Open source FiftyOne from public PyPI (multimodal datasets need fiftyone>=1.21)
!pip install -q "fiftyone>=1.21"

## Step 1: Load DROID and find the seed failure

Two public versions of the same DROID multimodal data (CC-BY-4.0, derived from [DROID](https://droid-dataset.github.io)):

| Repo | Episodes | Size |
|---|---|---|
| `dgural/droid-mcap-workshop` | 10 | 0.8 GB |
| `dgural/droid-mcap-demo` | 100 | 10.4 GB |

Both contain the seed episode. The default is the small one; switch `REPO` for the full set.

In [ ]:
import fiftyone.core.tags as fota
from fiftyone import ViewField as F
from fiftyone.utils.huggingface import load_from_hub

REPO = "dgural/droid-mcap-workshop"  # or "dgural/droid-mcap-demo" (100 episodes)

dataset = load_from_hub(REPO, name="droid-failure-mining", overwrite=True)
print(dataset)

In [ ]:
# Every episode has task and outcome fields
print(dataset.count_values("success"))
print(dataset.count_values("current_task"))

Open the App and look at the seed: **`AUTOLab+44bb9c36+2023-11-26-15h-53m-47s`** ("Fold cloth in half."). Every sensor plays on a shared clock. At about **6.6 s** the cloth corner slips out of the gripper before the fold completes.

Notice the dataset's automatic phase tags (`grasp`, `release`): the drop is labeled a normal `release`.

In [ ]:
SEED = "AUTOLab+44bb9c36+2023-11-26-15h-53m-47s.mcap"
seed = dataset.match(F("filepath").ends_with(SEED)).first()

# Existing phase tags on the seed (start/end are ns from the start of the recording)
for t in dataset.temporal_tags.values(
    filter=fota.TemporalTagFilter(sample_ids=[seed.id])
):
    print(f"{t.tag:<10} {t.start / 1e9:.2f}-{t.end / 1e9:.2f}s")

In [ ]:
# Optional: browse in the App
# session = fo.launch_app(dataset)

Record the failure with a temporal tag. In the App: Shift+drag across the timeline and type the tag. In code:

In [ ]:
TAG = "failure: dropped cloth"
NS = 1_000_000_000

dataset.temporal_tags.add(
    fota.TemporalTag(seed.id, start=int(6.60 * NS), end=int(7.07 * NS), tag=TAG)
)
print(dataset.temporal_tags.count())

## Step 2 (🚀 FiftyOne Enterprise): one failure, many similar cases

FiftyOne Enterprise embeds every short window of every camera stream (the blog uses 5 s windows with `qwen3-vl-embedding-2b-torch`), then searches by moment:

1. Open the **Embeddings** panel and pick the run.
2. Set **Streams** to one camera (e.g. `/camera/ext1/image_rgb`); otherwise the same moment on the robot's other cameras ranks first.
3. Clear grid filters (the search is scoped to the current view), click the seed's dropped-cloth window, and choose **Find similar**.

On the 100-episode set, one seed returned 16 similar cloth-handling episodes.

To build the index on your own data, run the **Compute similarity** operator from the App's operator browser on a multimodal dataset. It defaults to `qwen3-vl-embedding-2b-torch` with 5 s windows and runs as a delegated operation on your deployment's workers.


## Step 3: Narrow the list to real failures

Similarity search finds similar *situations*, not failures: an accidental drop and a deliberate placement look alike. Combine it with metadata you already have. DROID records each episode's outcome, so restrict review to failed episodes (in Enterprise, apply this view before **Find similar**):

In [ ]:
failed = dataset.match(F("success") == False)  # noqa: E712
print(f"{len(failed)} failed episodes to review")
for s in failed.select_fields(["current_task"]):
    print(f"  {s.filepath.split('/')[-1]}  {s.current_task.splitlines()[0]}")

Review each candidate with synced playback and temporal-tag the real failures, the same way as the seed. For example, on the 100-episode set the scoped search also surfaced `AUTOLab+44bb9c36+2023-11-26-16h-48m-21s` (cloth slips at about 11.2 s):

In [ ]:
HIT = "AUTOLab+44bb9c36+2023-11-26-16h-48m-21s.mcap"
hit = dataset.match(F("filepath").ends_with(HIT))
if len(hit):
    dataset.temporal_tags.add(
        fota.TemporalTag(
            hit.first().id, start=int(11.22 * NS), end=int(11.58 * NS), tag=TAG
        )
    )
else:
    print(f"{HIT} is only in the 100-episode set; skipping")

## Step 4: Save the curated failure set

In [ ]:
view = dataset.match_temporal_tags(tags=TAG)
dataset.save_view("failure-mining: dropped cloth", view, overwrite=True)
print(f"{len(view)} episodes with '{TAG}'")

In [ ]:
# Export the tags for these episodes (failure intervals plus their phase tags)
n = fota.export_tags(view, "dropped_cloth_tags.json")
print(f"exported {n} temporal tags")

## Next steps

- Repeat with a new seed: find a failure, search, narrow with metadata, review, tag.
- Take the curated set into triage and retraining (see the triage workflow post).